In [15]:
!python dataclean.py

Data Cleaning — Gamage Recruiters Dataset

Raw data loaded successfully.
      Rows    : 112
      Columns : ['Mention ID', 'Platform', 'Date', 'Author (Optional)', 'Caption of the post', 'Comment Text', 'No. of Likes', 'No. of Shares / Reposts', 'No. of Comments', 'Post URL', 'Type of Post']

Column names renamed to code-friendly format.
Before : 112 rows
After  : 112 rows
Removed: 0 empty rows

Platform names standardized.
      Unique platforms now: ['LinkedIn']

WARNING — 1 suspicious date(s) found and set to NaT:
   mention_id       date
14     LI-015 2036-02-26

Date range: 2025-01-04 -> 2026-12-02

Post type column fixed.
      Post type counts:
post_type
Marketing Post    64
Job Post          48

Language detection complete.
      Language breakdown:
language
english    104
sinhala      8
      NOTE: Sinhala posts will be excluded from English sentiment analysis.

Text cleaning applied to caption and comment columns.

      Example — BEFORE cleaning:
      In a competitive mark

Traceback (most recent call last):
  File "F:\physical science\4th year\Internship\Task 10\gamage-recruiters-sentiment-analysis\dataclean.py", line 313, in <module>
    df_output.to_excel(OUTPUT_FILE, index=False, sheet_name="Cleaned_Data")
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\pandas\core\generic.py", line 2439, in to_excel
    formatter.write(
    ~~~~~~~~~~~~~~~^
        excel_writer,
        ^^^^^^^^^^^^^
    ...<6 lines>...
        engine_kwargs=engine_kwargs,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\pandas\io\formats\excel.py", line 943, in write
    writer = ExcelWriter(
        writer,
    ...<2 lines>...
        engine_kwargs=engine_kwargs,
    )


In [13]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

print('Libraries imported successfully!')

Libraries imported successfully!


In [14]:
# Update this path to where your cleaned file is saved
INPUT_FILE  = 'Gamage_Recruiters_Cleaned_Data_Collection.xlsx'
OUTPUT_FILE = 'Gamage_Recruiters_Sentiment_Results.xlsx'

df = pd.read_excel(INPUT_FILE, sheet_name='Cleaned_Data')

print(f'Total rows loaded : {len(df)}')
print(f'Language breakdown:')
print(df['language'].value_counts())

# Separate Sinhala rows — VADER and TextBlob are English only
df_sinhala = df[df['language'] == 'sinhala'].copy()
df         = df[df['language'] == 'english'].copy()
df         = df.reset_index(drop=True)

print(f'\nEnglish rows for analysis : {len(df)}')
print(f'Sinhala rows (excluded)   : {len(df_sinhala)}')
df[['mention_id', 'post_type', 'text_for_analysis']].head(3)

Total rows loaded : 112
Language breakdown:
language
english    104
sinhala      8
Name: count, dtype: int64

English rows for analysis : 104
Sinhala rows (excluded)   : 8


,mention_id,post_type,text_for_analysis
0,LI-003,Marketing Post,"in a competitive marketplace, your success dep..."
1,LI-004,Marketing Post,in today s fast-paced and ever-evolving busine...
2,LI-005,Job Post,gamage recruiters pvt ltd we are recruiting! p...


In [16]:
# Initialize VADER
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    if not isinstance(text, str) or text.strip() == '':
        return {'vader_compound': 0.0, 'vader_pos': 0.0,
                'vader_neu': 1.0,  'vader_neg': 0.0, 'vader_label': 'Neutral'}
    
    scores   = vader.polarity_scores(text)
    compound = scores['compound']
    
    if compound >= 0.05:
        label = 'Positive'
    elif compound <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'
    
    return {
        'vader_compound': round(compound, 4),
        'vader_pos'     : round(scores['pos'], 4),
        'vader_neu'     : round(scores['neu'], 4),
        'vader_neg'     : round(scores['neg'], 4),
        'vader_label'   : label
    }

# Apply to every row
vader_results = df['text_for_analysis'].apply(get_vader_scores)
vader_df      = pd.json_normalize(vader_results)
df            = pd.concat([df, vader_df], axis=1)

print('VADER Analysis Complete!')
print('\nVADER Sentiment Distribution:')
print(df['vader_label'].value_counts())
df[['mention_id', 'text_for_analysis', 'vader_compound', 'vader_label']].head(5)

VADER Analysis Complete!

VADER Sentiment Distribution:
vader_label
Positive    94
Neutral      8
Negative     2
Name: count, dtype: int64


,mention_id,text_for_analysis,vader_compound,vader_label
0,LI-003,"in a competitive marketplace, your success dep...",0.9970,Positive
1,LI-004,in today s fast-paced and ever-evolving busine...,0.9932,Positive
2,LI-005,gamage recruiters pvt ltd we are recruiting! p...,0.7901,Positive
3,LI-006,gamage recruiters pvt ltd we are recruiting! p...,0.8356,Positive
4,LI-007,gamage recruiters pvt ltd we are recruiting! p...,-0.3802,Negative


In [17]:
def get_textblob_scores(text):
    if not isinstance(text, str) or text.strip() == '':
        return {'tb_polarity': 0.0, 'tb_subjectivity': 0.0, 'tb_label': 'Neutral'}
    
    blob         = TextBlob(text)
    polarity     = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    
    if polarity > 0:
        label = 'Positive'
    elif polarity < 0:
        label = 'Negative'
    else:
        label = 'Neutral'
    
    return {
        'tb_polarity'    : round(polarity, 4),
        'tb_subjectivity': round(subjectivity, 4),
        'tb_label'       : label
    }

# Apply to every row
tb_results = df['text_for_analysis'].apply(get_textblob_scores)
tb_df      = pd.json_normalize(tb_results)
df         = pd.concat([df, tb_df], axis=1)

print('TextBlob Analysis Complete!')
print('\nTextBlob Sentiment Distribution:')
print(df['tb_label'].value_counts())
df[['mention_id', 'text_for_analysis', 'tb_polarity', 'tb_subjectivity', 'tb_label']].head(5)

TextBlob Analysis Complete!

TextBlob Sentiment Distribution:
tb_label
Positive    93
Neutral     10
Negative     1
Name: count, dtype: int64


,mention_id,text_for_analysis,tb_polarity,tb_subjectivity,tb_label
0,LI-003,"in a competitive marketplace, your success dep...",0.3207,0.4276,Positive
1,LI-004,in today s fast-paced and ever-evolving busine...,0.3113,0.4884,Positive
2,LI-005,gamage recruiters pvt ltd we are recruiting! p...,0.2420,0.4080,Positive
3,LI-006,gamage recruiters pvt ltd we are recruiting! p...,0.1083,0.4333,Positive
4,LI-007,gamage recruiters pvt ltd we are recruiting! p...,0.0810,0.4286,Positive


In [18]:
df['methods_agree'] = df['vader_label'] == df['tb_label']

agree_count = df['methods_agree'].sum()
agree_pct   = round(agree_count / len(df) * 100, 1)

print(f'Rows where both agree    : {agree_count}  ({agree_pct}%)')
print(f'Rows where they disagree : {(~df["methods_agree"]).sum()}')
print()
print('Cross-tabulation (VADER vs TextBlob):')
print(pd.crosstab(df['vader_label'], df['tb_label'],
                  rownames=['VADER'], colnames=['TextBlob'], margins=True))

Rows where both agree    : 99  (95.2%)
Rows where they disagree : 5

Cross-tabulation (VADER vs TextBlob):
TextBlob  Negative  Neutral  Positive  All
VADER                                     
Negative         1        0         1    2
Neutral          0        7         1    8
Positive         0        3        91   94
All              1       10        93  104


In [19]:
def assign_final_sentiment(row):
    if row['vader_label'] == row['tb_label']:
        return row['vader_label'], 'High'
    else:
        return row['vader_label'], 'Low'   # VADER wins tiebreak

df[['final_sentiment', 'confidence']] = df.apply(
    assign_final_sentiment, axis=1, result_type='expand'
)

print('Final Sentiment Distribution:')
print('=' * 40)
counts = df['final_sentiment'].value_counts()
total  = len(df)
for label, count in counts.items():
    pct = round(count / total * 100, 1)
    bar = '█' * int(pct / 2)
    print(f'  {label:<10} : {count:>3} rows ({pct:>5}%)  {bar}')

print(f'\n  High confidence: {(df["confidence"] == "High").sum()} rows')
print(f'  Low confidence : {(df["confidence"] == "Low").sum()} rows')

Final Sentiment Distribution:
  Positive   :  94 rows ( 90.4%)  █████████████████████████████████████████████
  Neutral    :   8 rows (  7.7%)  ███
  Negative   :   2 rows (  1.9%)  

  High confidence: 99 rows
  Low confidence : 5 rows


In [20]:
print('Sentiment by Post Type')
print('=' * 50)
print(pd.crosstab(df['post_type'], df['final_sentiment'],
                  margins=True, margins_name='Total'))

print('\nPercentages by Post Type:')
pct = pd.crosstab(df['post_type'], df['final_sentiment'],
                  normalize='index').round(3) * 100
print(pct)

Sentiment by Post Type
final_sentiment  Negative  Neutral  Positive  Total
post_type                                          
Job Post                1        8        39     48
Marketing Post          1        0        55     56
Total                   2        8        94    104

Percentages by Post Type:
final_sentiment  Negative  Neutral  Positive
post_type                                   
Job Post              2.1     16.7      81.2
Marketing Post        1.8      0.0      98.2


In [21]:
for sentiment in ['Positive', 'Neutral', 'Negative']:
    subset = df[df['final_sentiment'] == sentiment]
    print(f'\n{"=" * 60}')
    print(f'  {sentiment.upper()}  ({len(subset)} total rows)')
    print(f'{"=" * 60}')
    for _, row in subset.head(2).iterrows():
        print(f'\n  ID          : {row["mention_id"]}')
        print(f'  VADER Score : {row["vader_compound"]}  → {row["vader_label"]}')
        print(f'  TextBlob    : {row["tb_polarity"]}  → {row["tb_label"]}')
        print(f'  Final Label : {row["final_sentiment"]}  (Confidence: {row["confidence"]})')
        print(f'  Text        : {str(row["text_for_analysis"])[:200]}...')


  POSITIVE  (94 total rows)

  ID          : LI-003
  VADER Score : 0.997  → Positive
  TextBlob    : 0.3207  → Positive
  Final Label : Positive  (Confidence: High)
  Text        : in a competitive marketplace, your success depends on one critical factor your people. but finding the right talent and managing them effectively takes more than just posting a job ad. it takes strate...

  ID          : LI-004
  VADER Score : 0.9932  → Positive
  TextBlob    : 0.3113  → Positive
  Final Label : Positive  (Confidence: High)
  Text        : in today s fast-paced and ever-evolving business landscape, your greatest asset is your people. but building a high-performing, engaged, and future-ready workforce doesn t happen by accident it takes ...

  NEUTRAL  (8 total rows)

  ID          : LI-018
  VADER Score : 0.0  → Neutral
  TextBlob    : 0.0  → Neutral
  Final Label : Neutral  (Confidence: High)
  Text        : gamage recruiters we're hiring! open positions civil engineer import export execu

In [23]:
output_cols = [
    'mention_id', 'platform', 'date', 'author', 'post_type', 'language',
    'text_for_analysis',
    'vader_compound', 'vader_pos', 'vader_neu', 'vader_neg', 'vader_label',
    'tb_polarity', 'tb_subjectivity', 'tb_label',
    'final_sentiment', 'confidence', 'methods_agree',
    'likes', 'shares', 'num_comments', 'post_url'
]

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df[output_cols].to_excel(writer, sheet_name='Sentiment_Results', index=False)
    df_sinhala.to_excel(writer, sheet_name='Sinhala_Posts', index=False)

print('=' * 55)
print('STEP 5 COMPLETE')
print('=' * 55)
print(f'  File saved : {OUTPUT_FILE}')
print(f'  Total rows : {len(df)}')
print()
counts = df['final_sentiment'].value_counts()
for label, count in counts.items():
    print(f'  {label:<10} : {count} ({round(count/len(df)*100, 1)}%)')


STEP 5 COMPLETE
  File saved : Gamage_Recruiters_Sentiment_Results.xlsx
  Total rows : 104

  Positive   : 94 (90.4%)
  Neutral    : 8 (7.7%)
  Negative   : 2 (1.9%)
